# Wan 2.2 TI2V-5B: moving shots

Turns each shot's keyframe into a short moving clip (image-to-video),
conditioned on the keyframe image and guided by the shot's `motion_prompt`.

**Before running:** attach a GPU to this notebook (Settings -> Accelerator -> GPU T4 x2),
add your Hugging Face token as a Kaggle Secret named `HF_TOKEN` (Add-ons -> Secrets),
and attach a dataset containing `storyboard.json`, `timing.json` (optional but recommended),
and a `keyframes/` folder with one `{shot_id}.jpg` per shot - zip your local
`storyboard.json` + `outputs/timing.json` + `outputs/keyframes/` together and
create a Kaggle Dataset from that zip, then attach it below.

**Real, confirmed risk:** Wan2.2's own README states TI2V-5B needs >=24GB VRAM
even with every memory-saving flag enabled (its own example GPU is an RTX 4090).
A Kaggle T4 has 16GB. This may out-of-memory - and unlike other Wan2.2 tasks,
**ti2v-5B has no smaller `--size` to fall back to**: confirmed live via
`generate.py`'s own `SUPPORTED_SIZES` check, this task accepts only
`704*1280` and `1280*704` - both the same pixel count, just rotated. If
1280*704 OOMs, the only things left to try are lowering `--frame_num`
(shorter clips may need less peak memory - untested, worth an experiment), or
falling back to this project's `--fallback` Ken Burns path, which needs no
GPU at all and is already a complete, working deliverable. Either way, log
what you tried in FAILURES.md.

In [ ]:
# --- Configuration: adjust these to match your attached dataset ---
INPUT_DIR = "/kaggle/input/aicontentgeneration"  # <-- change to your dataset's slug
# ti2v-5B only supports two --size choices (confirmed live via generate.py's
# own SUPPORTED_SIZES assertion): 704*1280 or 1280*704 - same pixel count,
# just rotated. No smaller resolution exists for this task.
SIZE = "1280*704"
OUTPUT_DIR = "/kaggle/working/clips"
FPS = 24  # Wan 2.2's native frame rate
# If 1280*704 OOMs and you want to test whether fewer frames avoids it
# (untested - resolution can't be lowered for this task, but peak memory may
# still scale with frame count), set e.g. MAX_FRAMES = 25 and re-run. None
# uses each shot's full timing-driven frame count.
MAX_FRAMES = None

In [ ]:
# --- Install Wan 2.2 ---
# flash_attn frequently fails to build from source on managed notebook images
# (confirmed live: "Getting requirements to build wheel" errored out, which
# aborted the whole `pip install -r requirements.txt` before it got to
# installing ftfy/dashscope - neither of which needs a build step at all,
# and generate.py crashes on a missing ftfy long before any flash_attn-related
# code runs). Installed separately so a flash_attn build failure can't take
# the rest of the requirements down with it.
!git clone https://github.com/Wan-Video/Wan2.2.git
%cd Wan2.2
!pip install -r <(grep -v flash_attn requirements.txt)
!pip install flash_attn --no-build-isolation || echo "flash_attn failed to build - continuing without it"

# Wan2.2 is pure PyTorch and never imports jax/flax - but diffusers does an
# optional Flax-compatibility probe at import time if it finds them
# installed, and a managed notebook's pre-installed jax build can be
# incompatible with its pre-installed numpy build (confirmed live on Colab:
# AttributeError on np.dtypes.StringDType deep inside jax's own dtypes
# module). Removing them makes diffusers' is_flax_available() check
# short-circuit to False instead of crashing - harmless, since nothing here
# needs either package.
!pip uninstall -y flax jax jaxlib -q

# wan/__init__.py unconditionally imports WanS2V (speech-to-video - a
# feature this project doesn't use, only TI2V), which needs `decord` for
# video reading. requirements.txt doesn't reliably bring it in on this image
# (confirmed live on Colab: ModuleNotFoundError at `import wan` without this).
!pip install -q decord

In [ ]:
# --- Download the Wan2.2-TI2V-5B weights (uses your HF_TOKEN Kaggle Secret) ---
# `huggingface-cli` is deprecated and silently does nothing on current
# huggingface_hub releases (confirmed live: it prints "deprecated and no
# longer works" and the hint text it prints recommends `hf` instead) -
# using that instead. Pinned to a range (not a blind -U upgrade) because
# forcing the latest huggingface_hub (2.4.0 at the time of writing) breaks
# the pre-pinned versions transformers/diffusers/tokenizers/datasets already
# expect (confirmed live: pip's resolver reported all four as incompatible
# with 2.4.0) - this range is the intersection of what all of them accept.
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!pip install -q "huggingface_hub>=0.34,<1.0"
!hf download Wan-AI/Wan2.2-TI2V-5B --local-dir ./Wan2.2-TI2V-5B

In [ ]:
# --- Load the storyboard and (if present) narration-driven timing ---
import json

with open(f"{INPUT_DIR}/storyboard.json") as f:
    storyboard = json.load(f)
shots = storyboard["shots"]

timing_by_id = {}
timing_path = f"{INPUT_DIR}/timing.json"
if os.path.exists(timing_path):
    with open(timing_path) as f:
        timing_by_id = {t["id"]: t for t in json.load(f)["shots"]}

print(f"{len(shots)} shots loaded; timing.json {'found' if timing_by_id else 'NOT found - using planned duration_s'}")

In [ ]:
# --- Generate one clip per shot ---
# Resumable: already-rendered shots are skipped, so re-running after a crash
# or a Kaggle session timeout continues where it left off.
import subprocess
import time

os.makedirs(OUTPUT_DIR, exist_ok=True)
gpu_log_path = f"{OUTPUT_DIR}/gpu_log.json"
gpu_log = json.load(open(gpu_log_path)) if os.path.exists(gpu_log_path) else {}


def frames_for_duration(duration_s, fps=FPS):
    # Wan2.2 requires frame_num = 4n + 1
    n = round(duration_s * fps)
    k = round((n - 1) / 4)
    frame_num = max(4 * k + 1, 5)
    if MAX_FRAMES is not None:
        frame_num = min(frame_num, MAX_FRAMES)
    return frame_num


for shot in shots:
    shot_id = shot["id"]
    out_path = f"{OUTPUT_DIR}/{shot_id}.mp4"
    if os.path.exists(out_path):
        print(f"[skip] {shot_id} already rendered")
        continue

    keyframe_path = None
    for ext in ("jpg", "jpeg", "png"):
        candidate = f"{INPUT_DIR}/keyframes/{shot_id}.{ext}"
        if os.path.exists(candidate):
            keyframe_path = candidate
            break
    if keyframe_path is None:
        print(f"[WARN] no keyframe found for {shot_id} (looked for {shot_id}.jpg/.jpeg/.png) - skipping")
        continue

    duration_s = timing_by_id.get(shot_id, {}).get("effective_duration_s", shot["duration_s"])
    frame_num = frames_for_duration(duration_s)
    prompt = shot["motion_prompt"]

    cmd = [
        "python", "generate.py",
        "--task", "ti2v-5B",
        "--size", SIZE,
        "--ckpt_dir", "./Wan2.2-TI2V-5B",
        "--offload_model", "True",
        "--convert_model_dtype",
        "--t5_cpu",
        "--image", keyframe_path,
        "--prompt", prompt,
        "--frame_num", str(frame_num),
        "--save_file", out_path,
    ]
    print(f"Generating {shot_id} ({frame_num} frames, ~{duration_s:.1f}s): {prompt!r}")
    start = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed_s = time.time() - start

    if result.returncode != 0:
        print(f"[ERROR] {shot_id} failed after {elapsed_s/60:.1f} min (returncode={result.returncode}):")
        print(result.stderr[-3000:])
        # A negative returncode means the process was killed by a signal
        # rather than raising a Python exception (no traceback printed at
        # all is the tell) - on Linux that's almost always -9 (SIGKILL) from
        # the kernel's OOM killer, consistent with this project's own
        # documented >=24GB VRAM risk on a 16GB T4. ti2v-5B has no smaller
        # --size to fall back to (confirmed live - see the markdown cell at
        # the top), so the only levers left are MAX_FRAMES above (untested)
        # or --fallback.
        if result.returncode < 0:
            print(f"[likely OOM] killed by signal {-result.returncode} - try setting MAX_FRAMES above (e.g. 25) and re-run this cell, or use this project's --fallback path instead.")
        else:
            print("If this looks like an out-of-memory error, try setting MAX_FRAMES above (e.g. 25) and re-run this cell, or use this project's --fallback path instead.")
        continue

    gpu_log[shot_id] = {"gpu_minutes": round(elapsed_s / 60, 2), "size": SIZE, "frame_num": frame_num}
    json.dump(gpu_log, open(gpu_log_path, "w"), indent=2)
    print(f"[OK] {shot_id} done in {elapsed_s/60:.1f} GPU minutes")

print("Done.")

In [ ]:
# --- Zip the clips for download ---
import shutil

shutil.make_archive("/kaggle/working/wan_clips", "zip", OUTPUT_DIR)
print("Saved /kaggle/working/wan_clips.zip - download it from the notebook's Output panel.")